### Очистка текста

In [57]:
text_data = [
    "    Interrobang. Ву Aishwaгya Нenriette     ",
    "Paгking And Going. Ву Кагl Gautier",
    "      Today Is The night. Ву Jarek Prakash      ",
]
strip_whitespace = [string.strip() for string in text_data]
print(strip_whitespace)

# Delete dots
remove_periods = [string.replace(".", "") for string in strip_whitespace]
print(remove_periods)

['Interrobang. Ву Aishwaгya Нenriette', 'Paгking And Going. Ву Кагl Gautier', 'Today Is The night. Ву Jarek Prakash']
['Interrobang Ву Aishwaгya Нenriette', 'Paгking And Going Ву Кагl Gautier', 'Today Is The night Ву Jarek Prakash']


### Parsing and cleaning HTML documents

In [58]:
from bs4 import BeautifulSoup

html = "<div class='full_name'>"\
"<span style='font-weight:bold'>Masego"\
"</span> Azra</div>"
soup = BeautifulSoup(html, "html.parser")

full_name_tag = soup.find("div", {"class": "full_name"})
full_name_tag.get_text(" ", strip=True) if full_name_tag else ""

'Masego Azra'

### Removing punctuation marks

In [59]:
import unicodedata
import sys

text_data = [
    "Ht!!!! I. Love. This. Song .... ",
    "10000% Agree!!!! #LoveIT",
    "Right?!?!",
]

punctuation = dict.fromkeys(
    (i for i in range(sys.maxunicode)
        if unicodedata.category(chr(i)).startswith('P')
    ),
    None
)

[string.translate(punctuation) for string in text_data]

['Ht I Love This Song  ', '10000 Agree LoveIT', 'Right']

### Text tokenization

In [60]:
from nltk.tokenize import word_tokenize
import nltk
from nltk.tokenize import sent_tokenize

string = "The science of today is the technology of tomorrow"
nltk.download('punkt_tab')
print(word_tokenize(string))

text_data = "The science of today is the technology of tomorrow. Tomorrow is today."
sent_tokenize(text_data)

['The', 'science', 'of', 'today', 'is', 'the', 'technology', 'of', 'tomorrow']


[nltk_data] Downloading package punkt_tab to /home/max/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


['The science of today is the technology of tomorrow.', 'Tomorrow is today.']

### Removing stop words

In [61]:
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

tokenized_words = [
    'i',
    'аm',
    'going',
    'to',
    'go',
    'to',
    'the',
    'store',
    'and',
    'park'
]

stop_words = stopwords.words("english")
[word for word in tokenized_words if word not in stop_words]


[nltk_data] Downloading package stopwords to /home/max/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


['аm', 'going', 'go', 'store', 'park']

### Word stemming

In [62]:
from nltk.stem.porter import PorterStemmer

tokenized_words = ['i', 'am', 'humbled', 'by', 'this', 'traditional', 'meeting']

porter = PorterStemmer()
[porter.stem(word) for word in tokenized_words]

['i', 'am', 'humbl', 'by', 'thi', 'tradit', 'meet']

### Разметка частей речи

In [63]:
from nltk import pos_tag
from nltk import word_tokenize

text_data = "Chris loved outdoor running"
try:
    text_tagged = pos_tag(word_tokenize(text_data))
except LookupError:
    nltk.download("averaged_perceptron_tagger_eng")
    text_tagged = pos_tag(word_tokenize(text_data))
    
text_tagged

# NNP Имя собственное, единственное число
# NN Существительное, единственное число
# RB Наречие
# VBD Глагол, прошедшее время
# VBN Глагол, причастие прошедшего времени
# VBG Глагол, причастие настоящего времени
# JJ Прилагательное
# PRP Местоимение

[('Chris', 'NNP'), ('loved', 'VBD'), ('outdoor', 'RP'), ('running', 'VBG')]

### Распознавание именованных сущностей

In [70]:
import spacy
from spacy.cli import download
try:
    spacy.load("en_core_web_sm")
except OSError:
    download("en_core_web_sm")
nlp = spacy.load("en_core_web_sm")
doc = nlp("Elon Musk offered to buy Twitter using $21B of his own money.")

print(doc.ents)

for entity in doc.ents:
    print(entity.text, entity.label_, sep=",")

(Elon Musk, Twitter, 21B)
Elon Musk,PERSON
Twitter,PERSON
21B,MONEY


### Representing text as a bag of words

In [76]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
text_data = np.array(
    ["I love Brazil. Brazil!", "Sweden is best", "Germany beats both"]
)

count = CountVectorizer()
bag_of_words = count.fit_transform(text_data)

bag_of_words.toarray()

count.get_feature_names_out()

array(['beats', 'best', 'both', 'brazil', 'germany', 'is', 'love',
       'sweden'], dtype=object)

### Assessing the importance of words

In [82]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
feature_matrix = tfidf.fit_transform(text_data)

feature_matrix.toarray()

tfidf.vocabulary_

{'love': 6,
 'brazil': 3,
 'sweden': 7,
 'is': 5,
 'best': 1,
 'germany': 4,
 'beats': 0,
 'both': 2}

### Using text vectorization to measure text-to-search query relevance

In [86]:
from sklearn.metrics.pairwise import linear_kernel
text = "Brazil is best"
vector = tfidf.transform([text])

cosine_similarities = linear_kernel(vector, feature_matrix).flatten()

related_doc_indices = cosine_similarities.argsort()[:-10:-1]

print([(text_data[i], cosine_similarities[i]) for i in related_doc_indices])

[(np.str_('Sweden is best'), np.float64(0.6666666666666666)), (np.str_('I love Brazil. Brazil!'), np.float64(0.5163977794943222)), (np.str_('Germany beats both'), np.float64(0.0))]
